# 🦅 E-gle Eye — Auto 10s Clip Saver

| 항목 | 내용 |
|------|------|
| **프로젝트** | E-gle Eye (Azure 기반 재난 대응 서포트 시스템) |
| **담당** | 로직 담당 (원준님) |
| **기능** | Red 진입 시 해당 카메라 10초 클립 자동 저장 |
| **우선순위** | ★★★★★ |
| **생성일** | 2026-03-03 |

---

## 📌 전체 구조

```
clips/
  camera_01/
    2026-03-01/
      21-16-05.mp4
  camera_02/
    ...

clip_log.jsonl   ← 클립 경로 + 메타데이터 자동 기록
```

### 흐름도
```
[RTSP 스트림]
     ↓  capture_loop()
[RingBuffer]  ← 15초 분량 JPEG 압축 프레임 상시 유지
     ↓  (Red 진입 이벤트 발생)
[save_10sec_clip_on_red()]
     ↓
[.mp4 저장]  +  [clip_log.jsonl 기록]  +  [Event Queue 전송]
```

## Cell 1 — 라이브러리 & 전역 설정

In [ ]:
# =====================================================
# Cell 1 : 라이브러리 & 전역 설정
# =====================================================
import cv2
import os
import time
import json
import queue
import datetime
import threading
import numpy as np
from collections import deque
from pathlib import Path

# ──────────────────────────────────────────
# ★ 원준님 설정값 (여기만 수정)
# ──────────────────────────────────────────
SAVE_ROOT       = "../clips"        # 클립 저장 루트 폴더
LOG_PATH        = "../clip_log.jsonl"  # 클립 로그 파일 (Event Dashboard 연동)
TARGET_FPS      = 12                # 8대 동시 운영 안전값
TARGET_WIDTH    = 640
TARGET_HEIGHT   = 360
JPEG_QUALITY    = 65                # 55~70 추천
BUFFER_SECONDS  = 15                # 10초 클립 + 5초 여유
MAX_FRAMES      = int(TARGET_FPS * BUFFER_SECONDS)  # 180 프레임
CLIP_DURATION   = 10.0              # 저장할 클립 길이 (초)
MIN_FRAMES_TO_SAVE = TARGET_FPS * 3 # 최소 버퍼 보장 프레임 수

# ──────────────────────────────────────────
# 전역 저장소
# ──────────────────────────────────────────
buffers         : dict = {}   # {camera_id: RingBuffer}
capture_threads : dict = {}   # {camera_id: Thread}
stop_flags      : dict = {}   # {camera_id: threading.Event}
event_queue     : queue.Queue = queue.Queue()   # Event Dashboard 연동 Queue

# 저장 폴더 생성
Path(SAVE_ROOT).mkdir(parents=True, exist_ok=True)
Path(LOG_PATH).parent.mkdir(parents=True, exist_ok=True)

print("✅ 설정 완료")
print(f"   저장 경로 : {SAVE_ROOT}")
print(f"   로그 파일 : {LOG_PATH}")
print(f"   버퍼 크기 : {MAX_FRAMES}프레임 ({BUFFER_SECONDS}초 @ {TARGET_FPS}fps)")

## Cell 2 — RingBuffer 클래스

In [ ]:
# =====================================================
# Cell 2 : RingBuffer — 링 버퍼 (자동 오래된 프레임 삭제)
# =====================================================
class RingBuffer:
    """
    JPEG 압축 바이트 + 타임스탬프를 쌍으로 관리하는 링 버퍼.
    maxlen 초과 시 가장 오래된 프레임 자동 삭제.
    """
    def __init__(self, maxlen: int):
        self.frames     = deque(maxlen=maxlen)
        self.timestamps = deque(maxlen=maxlen)
        self._lock      = threading.Lock()

    def add(self, encoded_frame: bytes, ts: float):
        """프레임 추가 (thread-safe)"""
        with self._lock:
            self.frames.append(encoded_frame)
            self.timestamps.append(ts)

    def __len__(self):
        return len(self.frames)

    def get_recent_clip(self, duration_sec: float = 10.0) -> list[bytes]:
        """
        최근 duration_sec 초 분량 프레임 반환 (시간 오름차순).
        thread-safe 복사 후 처리.
        """
        with self._lock:
            frames_snap = list(self.frames)
            ts_snap     = list(self.timestamps)

        if not ts_snap:
            return []

        cutoff = time.time() - duration_sec
        clip   = [
            frame
            for frame, ts in zip(frames_snap, ts_snap)
            if ts >= cutoff
        ]
        return clip

print("✅ RingBuffer 클래스 정의 완료")

## Cell 3 — 경로 & 로그 유틸리티

In [ ]:
# =====================================================
# Cell 3 : 경로 & 로그 유틸리티
# =====================================================

def get_clip_path(camera_id: str) -> str:
    """날짜별 폴더에 타임스탬프 파일명으로 클립 경로 생성"""
    now      = datetime.datetime.now()
    date_str = now.strftime("%Y-%m-%d")
    time_str = now.strftime("%H-%M-%S")
    folder   = os.path.join(SAVE_ROOT, camera_id, date_str)
    os.makedirs(folder, exist_ok=True)
    return os.path.join(folder, f"{time_str}.mp4")


_log_lock = threading.Lock()

def write_clip_log(
    camera_id : str,
    clip_path : str,
    frame_count : int,
    trigger   : str = "RED"
) -> None:
    """
    clip_log.jsonl 에 한 줄씩 기록.
    Event Dashboard가 이 파일을 tail하여 최신 클립 URL을 홈페이지에 표시.
    """
    entry = {
        "timestamp"   : datetime.datetime.now().isoformat(timespec="seconds"),
        "camera_id"   : camera_id,
        "trigger"     : trigger,
        "clip_path"   : clip_path,
        "frame_count" : frame_count,
        "duration_sec": round(frame_count / TARGET_FPS, 2),
    }
    with _log_lock:
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")


def tail_clip_log(n: int = 5) -> list[dict]:
    """clip_log.jsonl 마지막 n줄 반환 (홈페이지 최근 클립 조회용)"""
    if not os.path.exists(LOG_PATH):
        return []
    with open(LOG_PATH, "r", encoding="utf-8") as f:
        lines = f.readlines()
    return [json.loads(l) for l in lines[-n:]]


print("✅ 경로/로그 유틸리티 정의 완료")

## Cell 4 — 캡처 루프 & 스레드 관리

In [ ]:
# =====================================================
# Cell 4 : 캡처 루프 & 스레드 관리
# =====================================================

def capture_loop(camera_id: str, rtsp_url: str):
    """백그라운드 스레드: RTSP 프레임을 RingBuffer에 계속 적재"""
    cap = cv2.VideoCapture(rtsp_url)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 3)   # 버퍼 최소화 → 지연 감소

    if camera_id not in buffers:
        buffers[camera_id] = RingBuffer(MAX_FRAMES)

    print(f"[{camera_id}] ▶ 캡처 스레드 시작")
    interval       = 1.0 / TARGET_FPS
    last_frame_time = 0.0
    retry_count    = 0

    while not stop_flags.get(camera_id, threading.Event()).is_set():
        ret, frame = cap.read()

        if not ret:
            retry_count += 1
            wait = min(2 ** retry_count, 30)   # 지수 백오프, 최대 30초
            print(f"[{camera_id}] ⚠ 연결 끊김 (재시도 #{retry_count}) → {wait}초 후 재연결")
            time.sleep(wait)
            cap.open(rtsp_url)
            continue

        retry_count = 0  # 정상 수신 시 카운터 초기화
        now = time.time()

        if now - last_frame_time < interval:
            continue   # FPS 제한

        # 다운스케일 + JPEG 인코딩 (메모리 절약 핵심)
        small  = cv2.resize(frame, (TARGET_WIDTH, TARGET_HEIGHT))
        _, enc = cv2.imencode(
            ".jpg", small,
            [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY]
        )
        buffers[camera_id].add(enc.tobytes(), now)
        last_frame_time = now

    cap.release()
    print(f"[{camera_id}] ■ 캡처 스레드 종료")


def start_capture(camera_id: str, rtsp_url: str):
    """캡처 스레드 시작 (이미 실행 중이면 무시)"""
    if camera_id in capture_threads and capture_threads[camera_id].is_alive():
        print(f"[{camera_id}] ℹ 이미 실행 중")
        return
    buffers[camera_id]     = RingBuffer(MAX_FRAMES)   # 버퍼 초기화
    stop_flags[camera_id]  = threading.Event()
    t = threading.Thread(
        target=capture_loop,
        args=(camera_id, rtsp_url),
        daemon=True,
        name=f"capture-{camera_id}"
    )
    t.start()
    capture_threads[camera_id] = t
    print(f"[{camera_id}] ✅ 스레드 시작 완료")


def stop_capture(camera_id: str):
    """캡처 스레드 정지"""
    if camera_id in stop_flags:
        stop_flags[camera_id].set()
    if camera_id in capture_threads:
        capture_threads[camera_id].join(timeout=5)
        print(f"[{camera_id}] ■ 스레드 정지 완료")


def stop_all_captures():
    """모든 카메라 스레드 정지"""
    for cam_id in list(capture_threads.keys()):
        stop_capture(cam_id)
    print("■ 모든 캡처 스레드 정지")


print("✅ 캡처 루프 / 스레드 함수 정의 완료")

## Cell 5 — ★ 핵심: save_10sec_clip_on_red()

In [ ]:
# =====================================================
# Cell 5 : ★ 핵심 — Red 진입 시 10초 클립 저장
# =====================================================

def save_10sec_clip_on_red(
    camera_id    : str,
    trigger      : str  = "RED",
    push_to_queue: bool = True,
    confidence   : float | None = None
) -> str | None:
    """
    Red 진입 이벤트 발생 시 호출.
    RingBuffer에서 최근 10초 프레임을 꺼내 .mp4로 저장하고,
    ① clip_log.jsonl 에 경로 기록
    ② event_queue 에 이벤트 dict 전송 (Event Dashboard 연동)

    Parameters
    ----------
    camera_id     : 카메라 식별자 (예: "camera_01")
    trigger       : 트리거 레벨 ("RED" / "ORANGE" 등)
    push_to_queue : True면 event_queue에도 전송
    confidence    : AI 신뢰도 (0.0~1.0, 없으면 None)

    Returns
    -------
    저장된 클립 경로 (str) 또는 None (저장 실패)
    """
    # ── 1. 버퍼 유효성 검사 ──────────────────────────
    if camera_id not in buffers:
        print(f"[{camera_id}] ❌ 버퍼 없음 → 캡처가 시작되지 않았습니다")
        return None

    buf = buffers[camera_id]
    if len(buf) < MIN_FRAMES_TO_SAVE:
        print(f"[{camera_id}] ⚠ 버퍼 부족 ({len(buf)}/{MIN_FRAMES_TO_SAVE}프레임) → 클립 저장 스킵")
        return None

    # ── 2. 클립 프레임 추출 ──────────────────────────
    clip_frames = buf.get_recent_clip(duration_sec=CLIP_DURATION)
    if not clip_frames:
        print(f"[{camera_id}] ⚠ 추출된 프레임 없음")
        return None

    # ── 3. VideoWriter 초기화 및 저장 ────────────────
    clip_path   = get_clip_path(camera_id)
    first_frame = cv2.imdecode(
        np.frombuffer(clip_frames[0], np.uint8), cv2.IMREAD_COLOR
    )
    if first_frame is None:
        print(f"[{camera_id}] ❌ 첫 프레임 디코딩 실패")
        return None

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out    = cv2.VideoWriter(
        clip_path, fourcc, TARGET_FPS, (TARGET_WIDTH, TARGET_HEIGHT)
    )

    written = 0
    for encoded in clip_frames:
        frame = cv2.imdecode(np.frombuffer(encoded, np.uint8), cv2.IMREAD_COLOR)
        if frame is None:
            continue
        out.write(frame)
        written += 1

    out.release()

    # ── 4. 로그 기록 ─────────────────────────────────
    write_clip_log(
        camera_id   = camera_id,
        clip_path   = clip_path,
        frame_count = written,
        trigger     = trigger
    )

    # ── 5. Event Queue 전송 (Event Dashboard 연동) ───
    if push_to_queue:
        event = {
            "type"        : "clip_saved",
            "camera_id"   : camera_id,
            "trigger"     : trigger,
            "clip_path"   : clip_path,
            "frame_count" : written,
            "timestamp"   : datetime.datetime.now().isoformat(timespec="seconds"),
            "confidence"  : confidence,
        }
        event_queue.put(event)
        print(f"   → Event Queue 전송 완료 (qsize={event_queue.qsize()})")

    print(f"✅ [{camera_id}] {written}프레임 → {clip_path}")
    return clip_path


print("✅ save_10sec_clip_on_red() 정의 완료")

## Cell 6 — Event Dashboard 연동 헬퍼

In [ ]:
# =====================================================
# Cell 6 : Event Dashboard 연동 헬퍼
# =====================================================

def get_latest_clip_events(n: int = 10) -> list[dict]:
    """
    clip_log.jsonl에서 최근 n개 이벤트 반환.
    홈페이지 '최근 클립' 목록 API에서 호출.
    """
    return tail_clip_log(n)


def consume_event_queue(timeout: float = 0.1) -> dict | None:
    """
    Event Dashboard 스레드에서 큐에서 이벤트를 하나씩 꺼냄.
    Event Dashboard가 없다면 직접 polling하여 처리 가능.
    """
    try:
        return event_queue.get(timeout=timeout)
    except queue.Empty:
        return None


def print_clip_log_summary():
    """clip_log.jsonl 요약 출력 (디버깅용)"""
    entries = tail_clip_log(n=20)
    if not entries:
        print("(로그 없음)")
        return
    print(f"{'시각':<22} {'카메라':<12} {'트리거':<8} {'프레임':>6}  클립 경로")
    print("-" * 80)
    for e in entries:
        print(f"{e['timestamp']:<22} {e['camera_id']:<12} {e['trigger']:<8} "
              f"{e['frame_count']:>6}  {e['clip_path']}")


print("✅ Event Dashboard 헬퍼 정의 완료")

## Cell 7 — 오프라인 테스트 (RTSP 없이 로컬 영상으로 검증)

In [ ]:
# =====================================================
# Cell 7 : 오프라인 테스트 — 더미 프레임으로 버퍼 채우기
# RTSP 없이 기능 전체를 검증할 수 있습니다.
# =====================================================

TEST_CAMERA_ID = "camera_01"

def _fill_buffer_with_dummy_frames(camera_id: str, n_frames: int = 150):
    """테스트용: 컬러 그라데이션 더미 프레임으로 버퍼 채움"""
    if camera_id not in buffers:
        buffers[camera_id] = RingBuffer(MAX_FRAMES)
    buf = buffers[camera_id]
    base_time = time.time() - n_frames / TARGET_FPS
    for i in range(n_frames):
        color = int(255 * i / n_frames)
        frame = np.full((TARGET_HEIGHT, TARGET_WIDTH, 3), [color, 80, 200 - color], dtype=np.uint8)
        # 프레임 번호 텍스트 삽입
        cv2.putText(frame, f"CAM:{camera_id}  F:{i+1}/{n_frames}",
                    (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        _, enc = cv2.imencode(".jpg", frame, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])
        buf.add(enc.tobytes(), base_time + i / TARGET_FPS)
    print(f"[{camera_id}] 더미 프레임 {n_frames}개 버퍼 주입 완료")


# ─── 테스트 실행 ────────────────────────────────────
_fill_buffer_with_dummy_frames(TEST_CAMERA_ID, n_frames=150)
print(f"버퍼 현황: {len(buffers[TEST_CAMERA_ID])}프레임")

# Red 진입 시뮬레이션
saved = save_10sec_clip_on_red(
    camera_id     = TEST_CAMERA_ID,
    trigger       = "RED",
    confidence    = 0.93,
    push_to_queue = True
)

print()
print("─── 저장 결과 ───")
print(f"클립 경로: {saved}")
print(f"파일 존재: {os.path.exists(saved) if saved else False}")
if saved:
    size_kb = os.path.getsize(saved) / 1024
    print(f"파일 크기: {size_kb:.1f} KB")

print()
print("─── Event Queue 확인 ───")
event = consume_event_queue()
if event:
    for k, v in event.items():
        print(f"  {k:<14}: {v}")

print()
print("─── clip_log.jsonl 확인 ───")
print_clip_log_summary()

## Cell 8 — 실제 RTSP 연동 (운영 환경)

In [ ]:
# =====================================================
# Cell 8 : 실제 RTSP 연동 (운영 환경)
# camera_XX.json 에서 RTSP URL 읽어서 다중 카메라 시작
# =====================================================

# ─── 카메라 설정 (camera_XX.json 구조 예시) ─────────
# {
#   "camera_id": "camera_01",
#   "rtsp_url" : "rtsp://192.168.0.10:554/stream",
#   "location" : "1층 로비"
# }

def load_camera_config(config_path: str) -> dict:
    """camera_XX.json 에서 카메라 설정 로드"""
    with open(config_path, "r", encoding="utf-8") as f:
        return json.load(f)


def start_all_cameras_from_config(config_dir: str = "../configs"):
    """configs/ 폴더의 모든 camera_XX.json 파일로 캡처 시작"""
    config_dir = Path(config_dir)
    if not config_dir.exists():
        print(f"⚠ 설정 폴더 없음: {config_dir}")
        return
    configs = sorted(config_dir.glob("camera_*.json"))
    if not configs:
        print("⚠ camera_XX.json 파일이 없습니다.")
        return
    for cfg_path in configs:
        cfg = load_camera_config(str(cfg_path))
        start_capture(cfg["camera_id"], cfg["rtsp_url"])
        print(f"  [{cfg['camera_id']}] {cfg.get('location', '')} → {cfg['rtsp_url']}")
    print(f"\n총 {len(configs)}대 캡처 시작")


# ─── 단일 카메라 직접 시작 예시 ──────────────────────
# start_capture("camera_01", "rtsp://192.168.0.10:554/stream")

# ─── 전체 카메라 설정 파일로 시작 ────────────────────
# start_all_cameras_from_config("../configs")

# ─── Red 이벤트 수동 트리거 예시 ─────────────────────
# path = save_10sec_clip_on_red("camera_01", confidence=0.91)

print("✅ Cell 8 로드 완료 (주석 해제 후 사용)")

## Cell 9 — 상태 모니터링 & 종료

In [ ]:
# =====================================================
# Cell 9 : 상태 모니터링 & 종료
# =====================================================

def show_buffer_status():
    """현재 모든 카메라 버퍼 상태 출력"""
    print(f"{'카메라':<14} {'버퍼(프레임)':>12} {'버퍼(초)':>10} {'스레드 상태':<12}")
    print("-" * 55)
    for cam_id, buf in buffers.items():
        frames   = len(buf)
        secs     = round(frames / TARGET_FPS, 1)
        alive    = capture_threads.get(cam_id)
        status   = "🟢 실행중" if (alive and alive.is_alive()) else "🔴 중지"
        print(f"{cam_id:<14} {frames:>12} {secs:>9.1f}s  {status}")


# ─── 상태 확인 ────────────────────────────────────
show_buffer_status()

# ─── 모든 스레드 종료 (커널 재시작 전 필수) ──────────
# stop_all_captures()